# 11 - Beginner Seaborn Visualization with WEO Data

This lab focuses only on **Seaborn**. It uses the WEO Excel workbook and shows the most common Seaborn graph types in a beginner-friendly way.

You will learn how to choose the DataFrame, choose `x`, `y`, and grouping columns, then read the chart.

## Important: Use a Python Notebook Runtime

Run this notebook with a **Python 3** kernel in Jupyter, VS Code, JupyterLab, or Google Colab. Do not run these cells in SQL Server query mode.

This notebook uses the WEO Excel workbook from Module 4. Locally, it looks for `Module 4/labs/python-data-manipulation-database-connectivity/data/WEOApr2026all.xlsx`. In Google Colab, it also checks `/content/WEOApr2026all.xlsx` and can prompt you to upload the file if needed.

## 0. Install Packages

Seaborn depends on Matplotlib, and we also need pandas and openpyxl to read the Excel workbook.

In [ ]:
%pip install pandas numpy matplotlib seaborn openpyxl -q

## 1. Imports

In [ ]:
from pathlib import Path
from IPython.display import display
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

## 2. Display and Chart Settings

`pd.set_option(...)` only changes how pandas tables display in the notebook. `sns.set_theme(...)` changes the default Seaborn chart style.

In [ ]:
pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 160)

sns.set_theme(style="whitegrid", context="notebook")

OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

## 3. Find the WEO Workbook

In [ ]:
def find_weo_workbook():
    """Find the WEO workbook locally, in Colab, or through a Colab upload prompt."""
    candidate_paths = [
        Path("../../../Module 4/labs/python-data-manipulation-database-connectivity/data/WEOApr2026all.xlsx"),
        Path("Module 4/labs/python-data-manipulation-database-connectivity/data/WEOApr2026all.xlsx"),
        Path("data/WEOApr2026all.xlsx"),
        Path("/content/WEOApr2026all.xlsx"),
        Path("WEOApr2026all.xlsx"),
    ]

    for path in candidate_paths:
        if path.exists():
            return path

    try:
        from google.colab import files
        print("Upload WEOApr2026all.xlsx from Module 4 to continue.")
        uploaded = files.upload()
        if "WEOApr2026all.xlsx" in uploaded:
            return Path("WEOApr2026all.xlsx")
    except Exception:
        pass

    raise FileNotFoundError("WEOApr2026all.xlsx was not found.")


WEO_PATH = find_weo_workbook()
print("Using workbook:", WEO_PATH)

## 4. Load and Prepare the Data

The WEO workbook stores each year as a separate column. For visualization, we create:

- `country_macro`: one row per country and year, with useful indicators as columns.
- `latest`: the 2026 slice used for many country comparison charts.
- `group_long`: group-level trend data for line charts.

In [ ]:
INDICATOR_LABELS = {
    "NGDP_RPCH": "gdp_growth_pct",
    "PCPIPCH": "inflation_pct",
    "LUR": "unemployment_rate",
    "BCA_NGDPD": "current_account_pct_gdp",
    "GGXWDG_NGDP": "government_debt_pct_gdp",
    "NGDPDPC": "gdp_per_capita_usd",
    "NID_NGDP": "investment_pct_gdp",
    "NGSD_NGDP": "savings_pct_gdp",
    "TX_RPCH": "export_volume_growth_pct",
    "TM_RPCH": "import_volume_growth_pct",
}


def standardize_column_names(frame):
    """Convert non-year columns to snake_case names."""
    renamed_columns = {}
    for column in frame.columns:
        if isinstance(column, int):
            renamed_columns[column] = column
        else:
            renamed_columns[column] = str(column).strip().lower().replace(".", "_").replace(" ", "_").replace("-", "_")
    return frame.rename(columns=renamed_columns)


def get_year_columns(frame):
    """Find year columns such as 1980, 2024, and 2031."""
    return [column for column in frame.columns if isinstance(column, int) or str(column).isdigit()]


def make_long_indicator_data(frame, indicator_ids, source_sheet):
    """Convert wide WEO data into tidy long format."""
    year_columns = get_year_columns(frame)
    id_columns = ["country_id", "country", "indicator_id", "indicator", "unit"]
    filtered = frame.loc[frame["indicator_id"].isin(indicator_ids), id_columns + year_columns].copy()

    long_df = filtered.melt(
        id_vars=id_columns,
        value_vars=year_columns,
        var_name="year",
        value_name="value",
    )
    long_df["year"] = long_df["year"].astype(int)
    long_df["value"] = pd.to_numeric(long_df["value"], errors="coerce")
    long_df["source_sheet"] = source_sheet
    return long_df

In [ ]:
countries_raw = standardize_column_names(pd.read_excel(WEO_PATH, sheet_name="Countries"))
country_groups_raw = standardize_column_names(pd.read_excel(WEO_PATH, sheet_name="Country Groups"))
group_composition = standardize_column_names(pd.read_excel(WEO_PATH, sheet_name="Country Group Composition"))

group_composition = group_composition.rename(columns={
    "groupcode": "group_code",
    "groupname": "group_name",
    "countrycode": "country_id",
    "countryname": "country_name",
})

country_long = make_long_indicator_data(countries_raw, list(INDICATOR_LABELS), "Countries")

country_macro = (
    country_long.pivot_table(
        index=["country_id", "country", "year"],
        columns="indicator_id",
        values="value",
        aggfunc="first",
    )
    .reset_index()
    .rename(columns=INDICATOR_LABELS)
)
country_macro.columns.name = None

advanced_codes = set(group_composition.loc[group_composition["group_name"].eq("Advanced Economies"), "country_id"])
emerging_codes = set(group_composition.loc[group_composition["group_name"].eq("Emerging Market and Developing Economies"), "country_id"])

country_macro["economic_group"] = np.select(
    [country_macro["country_id"].isin(advanced_codes), country_macro["country_id"].isin(emerging_codes)],
    ["Advanced Economies", "Emerging Market and Developing Economies"],
    default="Other",
)

latest = country_macro[country_macro["year"].eq(2026)].copy()
group_long = make_long_indicator_data(country_groups_raw, ["NGDP_RPCH", "PCPIPCH"], "Country Groups")

print("Country-year rows:", len(country_macro))
print("Latest-year rows:", len(latest))
display(latest.head())

## 5. Seaborn Chart Pattern

Most Seaborn charts follow this structure:

```python
sns.chart_name(data=my_dataframe, x="column_for_x_axis", y="column_for_y_axis", hue="group_column")
```

The most important argument is `data=...`. That is the DataFrame being visualized.

## 6. Scatter Plot: Two Numeric Variables

Use `sns.scatterplot()` when each row should appear as one point. Here each point is a country.

In [ ]:
scatter_df = latest.dropna(subset=["inflation_pct", "gdp_growth_pct", "gdp_per_capita_usd"])

plt.figure(figsize=(8, 5))
sns.scatterplot(
    data=scatter_df,          # DataFrame used for the chart.
    x="inflation_pct",       # Horizontal axis.
    y="gdp_growth_pct",      # Vertical axis.
    hue="economic_group",    # Color by group.
    size="gdp_per_capita_usd",# Size by GDP per capita.
    sizes=(30, 220),
    alpha=0.75,
)
plt.title("Seaborn Scatter Plot: Inflation vs GDP Growth, 2026")
plt.xlabel("Inflation (%)")
plt.ylabel("GDP Growth (%)")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "weo_seaborn_scatterplot.png", dpi=150)
plt.show()

## 7. Line Plot: Trend Over Time

Use `sns.lineplot()` when the x-axis has time order.

In [ ]:
trend_groups = ["World", "Advanced Economies", "Emerging Market and Developing Economies", "Sub-Saharan Africa (SSA)"]
trend_df = group_long[
    group_long["country"].isin(trend_groups)
    & group_long["indicator_id"].eq("NGDP_RPCH")
    & group_long["year"].between(2015, 2031)
].copy()

plt.figure(figsize=(11, 5))
sns.lineplot(data=trend_df, x="year", y="value", hue="country", marker="o")
plt.title("Seaborn Line Plot: GDP Growth Trend by Group")
plt.xlabel("Year")
plt.ylabel("GDP Growth (%)")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "weo_seaborn_lineplot.png", dpi=150)
plt.show()

## 8. Histogram: Distribution of One Numeric Variable

Use `sns.histplot()` to see how one numeric column is distributed.

In [ ]:
plt.figure(figsize=(8, 4))
sns.histplot(data=latest, x="inflation_pct", bins=20, kde=False)
plt.title("Seaborn Histogram: Inflation Distribution, 2026")
plt.xlabel("Inflation (%)")
plt.ylabel("Number of Countries")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "weo_seaborn_histplot.png", dpi=150)
plt.show()

## 9. KDE Plot: Smooth Distribution Curve

Use `sns.kdeplot()` for a smooth estimate of a distribution. KDE is helpful for shape, but it is less literal than a histogram.

In [ ]:
plt.figure(figsize=(8, 4))
sns.kdeplot(data=latest.dropna(subset=["gdp_growth_pct"]), x="gdp_growth_pct", fill=True)
plt.title("Seaborn KDE Plot: GDP Growth Distribution, 2026")
plt.xlabel("GDP Growth (%)")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "weo_seaborn_kdeplot.png", dpi=150)
plt.show()

## 10. Bar Plot: Aggregated Category Comparison

Use `sns.barplot()` to compare an aggregate, such as the mean. Seaborn computes the average by default unless you provide already summarized data.

In [ ]:
bar_df = latest[latest["economic_group"].isin(["Advanced Economies", "Emerging Market and Developing Economies"])]

plt.figure(figsize=(8, 4))
sns.barplot(data=bar_df, x="economic_group", y="gdp_growth_pct", errorbar="sd")
plt.title("Seaborn Bar Plot: Average GDP Growth by Group, 2026")
plt.xlabel("Economic Group")
plt.ylabel("Average GDP Growth (%)")
plt.xticks(rotation=10)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "weo_seaborn_barplot.png", dpi=150)
plt.show()

## 11. Count Plot: Count Rows by Category

Use `sns.countplot()` to count how many rows are in each category. This does not need a `y` column.

In [ ]:
plt.figure(figsize=(8, 4))
sns.countplot(data=latest, x="economic_group", order=latest["economic_group"].value_counts().index)
plt.title("Seaborn Count Plot: Countries by Economic Group")
plt.xlabel("Economic Group")
plt.ylabel("Number of Countries")
plt.xticks(rotation=10)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "weo_seaborn_countplot.png", dpi=150)
plt.show()

## 12. Box Plot: Spread and Outliers

Use `sns.boxplot()` to compare spread and possible outliers across categories.

In [ ]:
plt.figure(figsize=(8, 4))
sns.boxplot(data=bar_df, x="economic_group", y="gdp_growth_pct")
plt.title("Seaborn Box Plot: GDP Growth Spread by Group")
plt.xlabel("Economic Group")
plt.ylabel("GDP Growth (%)")
plt.xticks(rotation=10)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "weo_seaborn_boxplot.png", dpi=150)
plt.show()

## 13. Violin Plot: Distribution Shape by Category

Use `sns.violinplot()` when you want a box-plot-like comparison plus a smoother view of distribution shape.

In [ ]:
plt.figure(figsize=(8, 4))
sns.violinplot(data=bar_df, x="economic_group", y="gdp_growth_pct", inner="quartile")
plt.title("Seaborn Violin Plot: GDP Growth Shape by Group")
plt.xlabel("Economic Group")
plt.ylabel("GDP Growth (%)")
plt.xticks(rotation=10)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "weo_seaborn_violinplot.png", dpi=150)
plt.show()

## 14. Strip Plot: Show Individual Points by Category

Use `sns.stripplot()` to show every observation in each category. Jitter spreads overlapping points sideways.

In [ ]:
plt.figure(figsize=(8, 4))
sns.stripplot(data=bar_df, x="economic_group", y="gdp_growth_pct", jitter=True, alpha=0.65)
plt.title("Seaborn Strip Plot: Country-Level GDP Growth Points")
plt.xlabel("Economic Group")
plt.ylabel("GDP Growth (%)")
plt.xticks(rotation=10)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "weo_seaborn_stripplot.png", dpi=150)
plt.show()

## 15. Regression Plot: Relationship with a Trend Line

Use `sns.regplot()` to add a fitted trend line to a scatter plot.

In [ ]:
reg_df = latest.dropna(subset=["inflation_pct", "gdp_growth_pct"])

plt.figure(figsize=(8, 5))
sns.regplot(data=reg_df, x="inflation_pct", y="gdp_growth_pct", scatter_kws={"alpha": 0.6})
plt.title("Seaborn Regression Plot: Inflation vs GDP Growth")
plt.xlabel("Inflation (%)")
plt.ylabel("GDP Growth (%)")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "weo_seaborn_regplot.png", dpi=150)
plt.show()

## 16. Heatmap: Correlation Matrix

Use `sns.heatmap()` to visualize a matrix. A correlation heatmap helps compare several numeric relationships at once.

In [ ]:
heatmap_columns = [
    "gdp_growth_pct",
    "inflation_pct",
    "unemployment_rate",
    "government_debt_pct_gdp",
    "current_account_pct_gdp",
    "investment_pct_gdp",
    "savings_pct_gdp",
    "gdp_per_capita_usd",
]

corr = latest[heatmap_columns].corr()

plt.figure(figsize=(9, 6))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", linewidths=0.5)
plt.title("Seaborn Heatmap: WEO Indicator Correlations, 2026")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "weo_seaborn_heatmap.png", dpi=150)
plt.show()

## 17. Pair Plot: Many Relationships at Once

Use `sns.pairplot()` for a quick matrix of scatter plots and distributions. Pair plots can become slow with many rows, so we use a small sample of columns.

In [ ]:
pair_columns = ["gdp_growth_pct", "inflation_pct", "government_debt_pct_gdp", "gdp_per_capita_usd", "economic_group"]
pair_df = latest[pair_columns].dropna().copy()

# Limit to the two main groups so the legend is easier to read.
pair_df = pair_df[pair_df["economic_group"].isin(["Advanced Economies", "Emerging Market and Developing Economies"])]

pair_grid = sns.pairplot(
    data=pair_df,
    vars=["gdp_growth_pct", "inflation_pct", "government_debt_pct_gdp", "gdp_per_capita_usd"],
    hue="economic_group",
    corner=True,
    plot_kws={"alpha": 0.65, "s": 30},
)
pair_grid.fig.suptitle("Seaborn Pair Plot: Selected WEO Indicators", y=1.02)
pair_grid.savefig(OUTPUT_DIR / "weo_seaborn_pairplot.png", dpi=150)
plt.show()

## 18. Facet Grid with `catplot`: Same Chart Split into Panels

Use `sns.catplot()` when you want Seaborn to create separate panels for a category.

In [ ]:
cat_df = country_macro[
    country_macro["year"].between(2024, 2026)
    & country_macro["economic_group"].isin(["Advanced Economies", "Emerging Market and Developing Economies"])
].copy()

cat_grid = sns.catplot(
    data=cat_df,
    x="year",
    y="gdp_growth_pct",
    col="economic_group",
    kind="box",
    height=4,
    aspect=1.1,
)
cat_grid.fig.suptitle("Seaborn Cat Plot: GDP Growth by Year and Group", y=1.05)
cat_grid.set_axis_labels("Year", "GDP Growth (%)")
cat_grid.savefig(OUTPUT_DIR / "weo_seaborn_catplot.png", dpi=150)
plt.show()

## 19. Seaborn Chart Selection Guide

Use this quick guide:

| Question | Seaborn chart |
| --- | --- |
| How do two numbers relate? | `scatterplot`, `regplot` |
| How does a value change over time? | `lineplot` |
| What is the distribution of one number? | `histplot`, `kdeplot` |
| How do groups compare by average? | `barplot` |
| How many rows are in each category? | `countplot` |
| How does spread differ by group? | `boxplot`, `violinplot`, `stripplot` |
| How do many numeric columns relate? | `heatmap`, `pairplot` |
| How can I split a chart into panels? | `catplot`, `FacetGrid` |